# Introduction

A long-slit spectrograph does something conceptually simple: it spreads a source's light out
by wavelength and records the result on a detector. What comes off the detector, though, is
not a spectrum. It is a two-dimensional image in which one axis runs roughly along wavelength,
the other runs along the slit, and neither is quite aligned with the pixel grid. The sky
contributes light everywhere. Nothing tells you which pixel corresponds to which wavelength.

Turning that image into a one-dimensional spectrum, flux against wavelength, is
**spectroscopic data reduction**, and it is what this book is about.

[`specreduce`](https://github.com/astropy/specreduce) is the Astropy-coordinated package for
that job. It supplies the building blocks: finding where the spectrum falls on the detector,
estimating and removing the sky background, collapsing the 2D image to 1D, and attaching a
wavelength scale derived from arc-lamp exposures. This book works through each block in turn
on real data, then composes them into a single scripted reduction.

## Two ways to use `specreduce`

`specreduce` is built for **both interactive work and automated pipelines**, and the same
objects serve both. Interactively you fit a trace, look at it, widen a background window,
check wavelength residuals by eye, and refit. In a pipeline you chain the identical calls
with no plots and no human in the loop.

The book teaches both, and in that order. Each step is introduced interactively, with figures,
so you can see what the operation does and how to tell when it has gone wrong. The final
chapter chains the whole sequence into a non-interactive reduction. Wavelength calibration is
the one step where the two styles differ enough to warrant separate chapters
([10](10-wavelength-solution-interactive.ipynb) and
[11](11-wavelength-solution-automated.ipynb)).

## How the book is organized

The chapters follow the order of an actual reduction, so each one starts where the previous
one finished.

**Foundations**

- [2. Spectral Images](02-spectral-images.ipynb): the dispersion and cross-dispersion axes,
  and how a frame is represented in code.
- [3. Spectrum Objects](03-spectrum-objects.ipynb): the `specutils` `Spectrum` containers
  that every `specreduce` operation consumes and returns.

**From frame to spectrum**

- [4. Tracing the Spectrum](04-tracing.ipynb): locating the source across the detector.
- [5. Arc Lamps](05-arc-lamps.ipynb): the wavelength-reference frames, and the distortion
  they make visible.
- [6. Tilt Correction](06-tilt-correction.ipynb): modeling and removing the tilt of spectral
  features across the slit, so that a column is a single wavelength. *Optional: whether you
  need it depends on your instrument.*
- [7. Background Subtraction](07-background.ipynb): estimating and removing the sky.
- [8. Spectral Extraction](08-extraction.ipynb): collapsing 2D to 1D, by summed aperture or
  optimal weighting.

**Wavelength calibration**

- [9. Line Identification](09-line-identification.ipynb): detecting arc emission lines and
  identifying them by hand against reference figures.
- [10. Interactive Workflow](10-wavelength-solution-interactive.ipynb) and
  [11. Automated Workflow](11-wavelength-solution-automated.ipynb): turning the identified
  lines into a wavelength solution by hand, and fitting one automatically from prior
  knowledge of the instrument setup.
- [12. The 2D Wavelength Solution](12-wavelength-solution-2d.ipynb): chaining the tilt and
  wavelength solutions into one object covering the whole detector.

**Flux calibration**

- [13. Flux Calibration](13-flux-calibration.ipynb): turning counts into physical flux. *A
  placeholder: the tools arrive in `specreduce` v1.11.*

**End to end**

- [14. Full Pipeline](14-full-pipeline.ipynb): every step chained into one reduction.

Tilt correction sits at chapter 6, before anything is measured from the frame, because
rectification rewrites the 2D image and every later step assumes wavelength does not vary along
the slit. That is the order a real reduction has to run in.


:::{note}
**Scope.** This book covers the steps `specreduce` itself performs. Detector-level reduction
(dark and flat correction, cosmic-ray rejection) belongs to
[`ccdproc`](https://ccdproc.readthedocs.io) and
[`astroscrappy`](https://astroscrappy.readthedocs.io) and is not taught here. Bias subtraction
appears only as plumbing: the frames below are raw, so the loader removes the bias pedestal
before handing them over.

Flux calibration is in scope but not yet demonstrated: `specreduce` gains the machinery for it
in **v1.11**, and [Chapter 13](13-flux-calibration.ipynb) is a placeholder until then. The book
therefore ends at a background-subtracted, extracted, wavelength-calibrated spectrum in
instrumental units.
:::

## The dataset

Every chapter works on the same data, so nothing has to be re-explained and each chapter can
pick up the previous one's output.

The frames come from [OSIRIS](https://www.gtc.iac.es/instruments/osiris/) on the 10.4 m
[Gran Telescopio Canarias](https://www.gtc.iac.es/) at the Roque de los Muchachos Observatory
on La Palma, taken in July 2014 with the R1000R grism. The science frame is a single 12-second
long-slit exposure of the TrES-3 system, drawn from a transmission-spectroscopy time series of
the transiting hot Jupiter TrES-3b. It comes with three arc-lamp exposures (HgAr, Ne, and
Xe) and a bias frame.

Two details of the setup are worth noticing now, because they shape later chapters. The science
exposure used a **40-arcsecond slit**, far wider than the star's seeing disc. That is
deliberate: in transmission spectroscopy the slit must not clip any light, or slit losses
masquerade as the tiny absorption signal you are trying to measure. The arcs, by contrast, were
taken through a 1.23-arcsecond slit, so their emission lines are sharp enough to calibrate
against: narrow lines for the wavelength solution, a wide slit for the science.

OSIRIS used two E2V CCD44-82 detectors (it was upgraded to OSIRIS+ in 2023). To keep the data
small, the frames here come from one CCD, binned and windowed to 512 × 1024 pixels: 512 along
the slit, 1024 along the dispersion.

A helper module keeps the loading in one place, so the chapters can get on with the reduction:

In [ ]:
import sys

sys.path.append("../src")

import matplotlib.pyplot as plt
from astropy.visualization import simple_norm

from common import read_data

plt.rc("figure", figsize=(8, 2.4), dpi=110)
plt.rc("font", size=9)

In [ ]:
arcs, lamps, obj = read_data(
    trim=False
)  # the frames as recorded; Chapter 2 explains the trim

frames = arcs + [obj]
labels = lamps + ("TrES-3 (science)",)

print(lamps)
print(obj.data.shape, obj.unit)

`read_data` returns the three arc frames and the science frame as
[`CCDData`](https://docs.astropy.org/en/stable/nddata/ccddata.html) objects, each carrying a
per-pixel variance. Three things happen on the way in, and all three read the FITS headers
rather than assuming anything:

- **The bias is subtracted.** These are raw frames, and every pixel sits on an electronic
  pedestal of roughly 4900 counts that has nothing to do with light.
- **Counts are converted to electrons**, using the `GAIN` keyword, 0.95 e⁻/ADU here.
  Analog-to-digital units are an arbitrary scale set by the readout electronics; electrons are
  the quantity photon statistics actually apply to, so this conversion has to happen before any
  uncertainty means anything.
- **A variance is attached**, combining photon shot noise with read noise. `specreduce`'s
  line-finding and optimal-extraction routines read the uncertainty to set detection thresholds
  and weight pixels, so a frame loaded without one quietly degrades everything downstream.

The read noise is *measured* rather than looked up. Each frame carries a `BIASSEC` keyword
pointing at its prescan: real pixels, clocked out and digitized like any other, but never
exposed to light. Once the bias is subtracted, all that remains there is noise, so its scatter
is the read noise, about 13 e⁻ per pixel for these frames. Measuring it also sidesteps a
subtlety that a catalog value would get wrong: these pixels were block-summed 2 × 2 when the
tutorial dataset was made, so each one is the sum of four independent reads and is noisier than
the detector's per-read figure. See `src/common.py` if you want the details.

Here is what the four frames look like:

In [ ]:
fig, axs = plt.subplots(4, figsize=(8, 6.5), sharex="all", constrained_layout=True)

for ax, frame, label in zip(axs, frames, labels):
    ax.imshow(
        frame.data,
        origin="lower",
        aspect="auto",
        cmap=plt.cm.Blues,
        norm=simple_norm(frame.data, stretch="log", vmin=0, vmax=250_000),
    )
    ax.text(0.01, 0.92, label, va="top", ha="left", transform=ax.transAxes)

plt.setp(axs, ylabel="Cross-dispersion [pix]")
plt.setp(axs[-1], xlabel="Dispersion [pix]");

The three arc frames are the wavelength reference. An arc lamp emits at a set of sharp,
laboratory-known wavelengths, so its exposure fills the slit with vertical lines whose true
wavelengths we can look up. Three lamps are used because no single one covers the range
usefully: HgAr is dense at the blue end, Xe at the red, Ne in between. Chapters
[5](05-arc-lamps.ipynb) and [9](09-line-identification.ipynb) through
[11](11-wavelength-solution-automated.ipynb) turn those lines into a wavelength scale.

The science frame looks almost empty by comparison. The bright horizontal band near
cross-dispersion row 275 is the target's spectrum, smeared along the dispersion axis; the
diffuse glow filling the rest of the slit is sky background. Look closely and you can pick out
several much fainter traces from other objects that happened to fall in the slit.

## What the raw frames need fixing

Three problems are visible in that figure, and they set up most of the book.

**The spectral lines are tilted.** They should be vertical, since a column of pixels ought to be a
single wavelength, but they lean and curve, increasingly so toward the edges of the slit.
This is an optical distortion, and it means wavelength varies along the cross-dispersion axis.
It is easier to see zoomed in:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3), constrained_layout=True)

ne = arcs[lamps.index("Ne")].data
ax.imshow(
    ne,
    origin="lower",
    aspect="auto",
    cmap=plt.cm.Blues,
    interpolation="bicubic",
    norm=simple_norm(ne, stretch="log", vmin=0, vmax=150_000),
)
ax.set(
    xlim=(250, 420),
    ylim=(20, 500),
    xlabel="Dispersion [pix]",
    ylabel="Cross-dispersion [pix]",
)
# Ticks on the top spine as well, so the lean can be read off at both ends of the slit.
ax.tick_params(axis="x", top=True, labeltop=True);

Across the full slit, a line shifts by several pixels. That is far more than the width of
the lines themselves, so it cannot be ignored: it would blur the wavelength solution and, worse,
corrupt sky subtraction, because the sky rows being subtracted from the target row would be at
slightly different wavelengths. In transit spectroscopy this is not a subtlety: published
analyses of OSIRIS transit data have traced spurious time-variability in telluric absorption
lines back to insufficient tilt correction, which is exactly the kind of systematic that can
imitate a planetary atmosphere.
[Chapter 6](06-tilt-correction.ipynb) fits a 2D model of the distortion and *rectifies* the
frames so that columns become wavelengths again. It does so early, because everything after it
assumes a column is a single wavelength.

**There is no wavelength scale.** Nothing in the frames says which pixel is which wavelength.
To make the point, here is a crude cut through the science trace, just the eleven rows around
row 275, summed:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 2.6), constrained_layout=True)

ax.plot(obj.data[270:281].sum(axis=0), lw=0.8)
ax.set(xlabel="Dispersion [pix]", ylabel="Signal [e$^-$]", xlim=(0, obj.shape[1]));

This is recognizably a spectrum. The broad hump is the combined response of the star, the
grism, and the detector, and the dips are real absorption, some stellar, some from Earth's
atmosphere. But the horizontal axis is pixels, so we cannot yet say *which* absorption features
these are, and the vertical axis includes sky background that we have not removed.

**The sky is still in there.** The cut above sums the target rows and the sky that lies on top
of them. [Chapter 7](07-background.ipynb) estimates the sky from rows on either side of the
trace and subtracts it, and [Chapter 8](08-extraction.ipynb) does the collapse to 1D properly,
weighting pixels by their signal-to-noise instead of summing a fixed block of rows.

By [Chapter 14](14-full-pipeline.ipynb) this same cut has become a background-subtracted,
optimally extracted spectrum on a calibrated wavelength axis, produced by a script rather than
by hand.

## Running these notebooks

Every chapter is an executable notebook. To run them yourself you will need `specreduce` and
its dependencies (see `requirements.txt` in the repository) plus the data files under
`data/`, which `common.read_data` locates automatically.

Chapters assume you have read the ones before them, and reuse the same variable names
(`arcs`, `lamps`, `obj`) throughout. Each chapter reloads the data from scratch, though, so you
can open any one of them on its own and run it top to bottom.

With that, on to the [2D spectral image](02-spectral-images.ipynb).